In [1]:
!pip install yfinance pandas numpy matplotlib scikit-learn

In [2]:
# importing librabries

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [6]:
# Download 5 years of price data for 5 stocks using yfinance

stocks = ["AAPL","MSFT","NVDA","GOOGL","AMZN"]
prices = yf.download(
    stocks,
    start = "2021-05-26",
    auto_adjust = True,
    progress = False
)["Open"] # type of price(Open,High,Low,Close,Volume)

prices.head()
    

Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2021-05-26,123.228127,162.925507,117.353089,240.643945,15.547065
2021-05-27,121.921965,161.501999,116.847259,239.204399,15.421133
2021-05-28,121.405362,160.985001,116.678145,239.501937,15.462029
2021-06-01,120.810762,160.453003,116.749070,237.006701,15.863266
2021-06-02,120.917972,160.399994,116.653849,235.931799,16.194682


In [8]:
# Calculate daily returns
returns = prices.pct_change().dropna() # calculates return (pt - pt-1)/pt-1
returns.head()

Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2021-05-27,-0.010600,-0.008737,-0.004310,-0.005982,-0.008100
2021-05-28,-0.004237,-0.003201,-0.001447,0.001244,0.002652
2021-06-01,-0.004898,-0.003305,0.000608,-0.010418,0.025950
2021-06-02,0.000887,-0.000330,-0.000816,-0.004535,0.020892
2021-06-03,-0.007416,-0.007472,-0.009709,-0.011552,0.021404


In [10]:
# create portfolio weights
weights = np.array([0.3, 0.2, 0.1, 0.2, 0.2])

portfolio_returns = returns.dot(weights) # dot product of returns and weights

# display first few portfolio returns
print(portfolio_returns.head())

Date
2021-05-27   -0.008175
2021-05-28   -0.001277
2021-06-01    0.001037
2021-06-02    0.003390
2021-06-03   -0.002720
dtype: float64


In [15]:
# Sort historical portfolio returns to find the VaR percentile
sorted_returns = portfolio_returns.sort_values()# 
print(sorted_returns)

# 5th percentile (95% VaR)
historical_var = np.percentile(portfolio_returns, 5)

print(historical_var)

Date
2024-08-05   -0.077318
2022-01-24   -0.061453
2025-04-04   -0.060437
2025-04-03   -0.054648
2022-09-13   -0.053734
                ...   
2022-10-04    0.052952
2023-05-25    0.053127
2023-02-02    0.058567
2022-05-27    0.061742
2022-02-25    0.075550
Length: 1254, dtype: float64
-0.025621003415124334


In [16]:
# Calculate CVaR as the average of returns below the VaR threshold

tail_losses = portfolio_returns[portfolio_returns <= historical_var]

historical_cvar = tail_losses.mean()

print(historical_cvar)

-0.03599370640257526


In [17]:
print(tail_losses.head())
print(len(tail_losses))

Date
2021-09-20   -0.038102
2021-12-14   -0.026802
2022-01-05   -0.026069
2022-01-10   -0.027929
2022-01-13   -0.027161
dtype: float64
63


In [18]:
# Compare historical vs parametric VaR 

mean_return = portfolio_returns.mean()
std_return = portfolio_returns.std()

print(mean_return)
print(std_return)

0.0011090996693750084
0.01585609607039292


In [19]:
parametric_var = mean_return - (1.645 * std_return)

print(parametric_var)

-0.024974178366421344


In [20]:
print("Historical VaR:", historical_var)
print("Parametric VaR:", parametric_var)

Historical VaR: -0.025621003415124334
Parametric VaR: -0.024974178366421344
